In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from db import query

In [ ]:
summary = query("SELECT * FROM detection_summary ORDER BY f1 DESC")
display(summary)

In [ ]:
df = query("SELECT * FROM bonferroni_fdr_results")

# Check if any row differs between the two methods
diff = df[df['bonferroni_rejected'] != df['fdr_rejected']]
print(f"Rows where Bonferroni != FDR: {len(diff)}")
display(diff)

# Rejection counts by group
print(df.groupby('feature_group')[['bonferroni_rejected','fdr_rejected']].sum())

In [ ]:
survived = df[(df['feature_group'] == 'spurious') & (df['bonferroni_rejected'] == False)]
display(survived[['feature','correlation','p_value']].sort_values('p_value'))

In [ ]:
flagged = df[(df['feature_group'] == 'control') & (df['bonferroni_rejected'] == True)]
display(flagged[['feature','correlation','p_value']])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, group in zip(axes, ['control', 'spurious', 'text']):
    subset = df[df['feature_group'] == group]['p_value']
    ax.hist(subset, bins=50, edgecolor='black')
    ax.set_title(f'{group} (n={len(subset)})')
    ax.set_xlabel('p-value')
plt.tight_layout()
plt.savefig('plots/pvalue_distributions.png', dpi=150)
plt.show()

In [ ]:
boot = query("SELECT * FROM bootstrap_results")
caught = boot[(boot['feature_group'] == 'spurious') & (boot['bootstrap_rejected'] == True)]
display(caught[['feature','insample_sharpe','critical_value_95']])

# Sharpe distribution by group
print(boot.groupby('feature_group')[['insample_sharpe','critical_value_95']].describe())

In [ ]:
wf = query("SELECT * FROM walkforward_results")

fig, ax = plt.subplots(figsize=(10, 5))
for group, color in [('control','green'), ('spurious','red'), ('text','blue')]:
    subset = wf[wf['feature_group'] == group]['mean_gen_ratio']
    ax.hist(subset, bins=50, alpha=0.5, label=group, color=color)
ax.axvline(0, color='black', linestyle='--', label='ratio=0')
ax.set_xlabel('Mean Generalization Ratio')
ax.set_title('Walk-Forward: Gen Ratio Distribution by Group')
ax.legend()
plt.savefig('plots/walkforward_distributions.png', dpi=150)
plt.show()